# Day 4 Capstone Lab - Memory Leak Detector
### AIOps Fundamentals Training | Microland

---

## The Full AIOps Loop: Observe - Think - Act

This is your capstone notebook. Unlike the previous labs, which focused on one skill at a time,
this notebook takes you through the **complete AIOps workflow** end to end.

By the end of this lab you will have:
1. **OBSERVED** — ingested 72 hours of realistic RAM utilisation data into Elastic Cloud
2. **THOUGHT** — used ML anomaly detection to identify a progressive memory leak automatically
3. **ACTED** — configured an automated webhook alert that fires a notification when the leak is detected

---

## Scenario

**Server:** `microland-app-03` — a Windows application server running a Java-based
middleware service for a client-facing portal.

**The Problem:** The application has a slow memory leak.
RAM utilisation has been rising at approximately **+2% per hour** for the past 24 hours.
It has not yet caused any visible performance degradation, so no user complaints have been raised.
If left unaddressed, the service will exhaust available memory and crash in approximately **18 hours**.

**Without AIOps:** This leak would go undetected until the service crashes, users call the helpdesk,
and an L1 engineer raises a P1 incident at 3 AM.

**With AIOps:** The ML detects the progressive deviation from baseline hours before the crash,
fires an automated alert, and gives the operations team time to schedule a controlled restart
during a maintenance window.

**Your task:** Build the detection and alerting system that catches this before it becomes an incident.

---

## Before You Start
1. Elastic Cloud deployment running (from Day 1)
2. Cloud ID and API Key ready
3. A free webhook URL from **webhook.site** (open it in a new tab now)
4. Running in Google Colab

> **No Python knowledge required.** Run each cell in order.
> This lab takes approximately 90 minutes including the Kibana steps.


## Step 1 - Install Required Libraries


In [ ]:
!pip install elasticsearch --quiet
print('Libraries installed.')


## Step 2 - Configuration

Fill in your Elastic Cloud credentials below.
The simulation parameters are pre-set to match the scenario — review them
before running so you understand exactly what data will be generated.


In [ ]:
# -----------------------------------------------------------
# ELASTIC CLOUD CREDENTIALS
# -----------------------------------------------------------
CLOUD_ID   = 'YOUR_CLOUD_ID_HERE'
API_KEY    = 'YOUR_API_KEY_HERE'
INDEX_NAME = 'aiops-capstone'

# -----------------------------------------------------------
# SCENARIO PARAMETERS
# -----------------------------------------------------------
SERVER_NAME        = 'microland-app-03'  # The affected server
TOTAL_HOURS        = 72                  # 3 days of data total
STABLE_HOURS       = 48                  # First 48 hours: stable baseline
LEAK_HOURS         = 24                  # Last 24 hours: progressive leak
SAMPLE_INTERVAL_S  = 30                  # One sample every 30 seconds

# Memory baseline: server sits at ~55% RAM utilisation during normal operation
BASELINE_MEAN      = 0.55
BASELINE_NOISE     = 0.03   # +/- 3% natural variation

# Leak profile: RAM rises +2% per hour during the leak phase
LEAK_RATE_PER_HOUR = 0.02   # 2% per hour
CRASH_THRESHOLD    = 0.90   # Service becomes unstable above 90% RAM

# Calculate when crash would occur
start_of_leak_pct  = BASELINE_MEAN
hours_to_crash     = (CRASH_THRESHOLD - start_of_leak_pct) / LEAK_RATE_PER_HOUR

print('Scenario configuration:')
print(f'   Server              : {SERVER_NAME}')
print(f'   Total data window   : {TOTAL_HOURS} hours ({TOTAL_HOURS // 24} days)')
print(f'   Stable baseline     : hours 0 - {STABLE_HOURS}')
print(f'   Leak phase begins   : hour {STABLE_HOURS}')
print(f'   Leak rate           : +{LEAK_RATE_PER_HOUR*100:.0f}% RAM per hour')
print(f'   Baseline RAM        : {BASELINE_MEAN*100:.0f}%')
print(f'   Crash threshold     : {CRASH_THRESHOLD*100:.0f}%')
print()
print(f'   Estimated time to crash from leak start : {hours_to_crash:.1f} hours')
print(f'   At current leak rate the service will crash ~{hours_to_crash:.0f}h after hour {STABLE_HOURS}')
total_samples = int(TOTAL_HOURS * 3600 / SAMPLE_INTERVAL_S)
print(f'   Total data points to generate : {total_samples:,}')


## Step 3 - Connect to Elastic Cloud


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
info = es.info()
print(f'Connected to cluster: {info["cluster_name"]}')
print(f'   Version: {info["version"]["number"]}')


## Step 4 - Create the Capstone Index

The capstone index stores memory metrics alongside a full suite of system metrics,
mirroring what Metricbeat ships from a real Windows server.
This richer schema lets the ML job consider multiple signals simultaneously.


In [ ]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':                 {'type': 'date'},
            'host.name':                  {'type': 'keyword'},
            'host.os.name':               {'type': 'keyword'},
            'host.os.version':            {'type': 'keyword'},
            # Memory metrics
            'system.memory.used.pct':     {'type': 'float'},
            'system.memory.used.bytes':   {'type': 'long'},
            'system.memory.free.bytes':   {'type': 'long'},
            'system.memory.total':        {'type': 'long'},
            'system.memory.actual.used.pct': {'type': 'float'},
            # CPU metrics (contextual signal)
            'system.cpu.total.pct':       {'type': 'float'},
            # Swap metrics (memory pressure indicator)
            'system.memory.swap.used.pct':{'type': 'float'},
            # Metadata
            'metricset.name':             {'type': 'keyword'},
            'event.dataset':              {'type': 'keyword'},
            'service.name':               {'type': 'keyword'},
            'phase':                      {'type': 'keyword'},
            'phase_description':          {'type': 'text'}
        }
    }
}

es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index "{INDEX_NAME}" created with full Metricbeat-compatible memory schema.')


## Step 5 - Generate the 72-Hour Memory Dataset

### Data Profile

| Phase | Duration | RAM Behaviour | Description |
|-------|----------|---------------|-------------|
| **Baseline** | Hours 0-48 | Stable at ~55% with small noise | Normal operation — the ML learns this as 'normal' |
| **Leak phase** | Hours 48-72 | Rising +2% per hour | Progressive memory leak — slow enough to miss without ML |

### Why This Matters

A 2% per hour rise is not visible on a single dashboard check.
At any given moment, the value looks 'high but acceptable'.
Only by comparing it to the learned baseline over time does the trend become apparent.
This is precisely what ML time-series anomaly detection is designed to catch.


In [ ]:
import random
import math
from datetime import datetime, timedelta, timezone

random.seed(2025)

now        = datetime.now(timezone.utc)
start_time = now - timedelta(hours=TOTAL_HOURS)
leak_start = start_time + timedelta(hours=STABLE_HOURS)

total_samples = int(TOTAL_HOURS * 3600 / SAMPLE_INTERVAL_S)
TOTAL_RAM_BYTES = 16 * 1024 * 1024 * 1024  # 16 GB RAM

events = []

for i in range(total_samples):
    ts      = start_time + timedelta(seconds=i * SAMPLE_INTERVAL_S)
    elapsed = i * SAMPLE_INTERVAL_S / 3600  # hours elapsed

    # Diurnal cycle: slight workload variation across the day (business hours busier)
    hour_of_day   = ts.hour + ts.minute / 60
    diurnal_delta = 0.03 * math.sin(math.pi * (hour_of_day - 6) / 12) if 6 <= hour_of_day <= 18 else 0.0

    if elapsed < STABLE_HOURS:
        # --- BASELINE PHASE ---
        mem_pct = (
            BASELINE_MEAN
            + diurnal_delta
            + random.gauss(0, BASELINE_NOISE)
        )
        mem_pct = min(0.72, max(0.45, mem_pct))
        phase        = 'baseline'
        phase_desc   = 'Normal operation: stable memory utilisation'
        swap_pct     = max(0.0, random.gauss(0.02, 0.01))  # minimal swap
    else:
        # --- LEAK PHASE ---
        hours_leaking = elapsed - STABLE_HOURS
        leak_increment = LEAK_RATE_PER_HOUR * hours_leaking
        # Noise increases slightly as pressure mounts (GC activity variation)
        leak_noise     = BASELINE_NOISE * (1 + hours_leaking / LEAK_HOURS)
        mem_pct = (
            BASELINE_MEAN
            + leak_increment
            + diurnal_delta
            + random.gauss(0, leak_noise)
        )
        mem_pct  = min(0.97, max(0.50, mem_pct))
        # Swap usage increases as RAM pressure builds
        swap_pct = min(0.60, max(0.0, (hours_leaking / LEAK_HOURS) * 0.40
                                 + random.gauss(0, 0.03)))
        phase        = 'leak'
        phase_desc   = (
            f'Memory leak detected: +{LEAK_RATE_PER_HOUR*100:.0f}% per hour. '
            f'Current leak duration: {hours_leaking:.1f}h. '
            f'Estimated time to crash threshold: {max(0, hours_to_crash - hours_leaking):.1f}h'
        )

    # CPU varies independently but trends slightly higher under memory pressure
    mem_pressure_on_cpu = 0.05 * max(0, mem_pct - BASELINE_MEAN - 0.10)
    cpu_pct = min(0.90, max(0.05,
                            0.30 + diurnal_delta * 0.5
                            + mem_pressure_on_cpu
                            + random.gauss(0, 0.06)))

    used_bytes = int(mem_pct * TOTAL_RAM_BYTES)
    free_bytes = TOTAL_RAM_BYTES - used_bytes

    events.append({
        '@timestamp':                    ts.isoformat(),
        'host.name':                     SERVER_NAME,
        'host.os.name':                  'Windows Server 2019',
        'host.os.version':               '10.0.17763',
        'system.memory.used.pct':        round(mem_pct, 4),
        'system.memory.actual.used.pct': round(mem_pct * 0.97, 4),
        'system.memory.used.bytes':      used_bytes,
        'system.memory.free.bytes':      free_bytes,
        'system.memory.total':           TOTAL_RAM_BYTES,
        'system.cpu.total.pct':          round(cpu_pct, 4),
        'system.memory.swap.used.pct':   round(swap_pct, 4),
        'metricset.name':                'memory',
        'event.dataset':                 'system.memory',
        'service.name':                  'java-middleware-svc',
        'phase':                         phase,
        'phase_description':             phase_desc
    })

baseline_pts = [e for e in events if e['phase'] == 'baseline']
leak_pts     = [e for e in events if e['phase'] == 'leak']

avg_baseline = sum(e['system.memory.used.pct'] for e in baseline_pts) / len(baseline_pts)
avg_leak     = sum(e['system.memory.used.pct'] for e in leak_pts)     / len(leak_pts)
peak_leak    = max(e['system.memory.used.pct'] for e in leak_pts)
final_val    = events[-1]['system.memory.used.pct']

print(f'Generated {len(events):,} memory metric samples')
print(f'   Sample interval        : every {SAMPLE_INTERVAL_S}s')
print(f'   Baseline samples       : {len(baseline_pts):,}  (hours 0-{STABLE_HOURS})')
print(f'   Leak phase samples     : {len(leak_pts):,}  (hours {STABLE_HOURS}-{TOTAL_HOURS})')
print()
print(f'Memory statistics:')
print(f'   Baseline average       : {avg_baseline*100:.1f}%')
print(f'   Leak phase average     : {avg_leak*100:.1f}%')
print(f'   Peak value (end)       : {peak_leak*100:.1f}%')
print(f'   Final data point       : {final_val*100:.1f}%')
print()
print(f'Scenario timeline:')
print(f'   Leak begins            : {leak_start.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'   Crash threshold (90%)  : ~{hours_to_crash:.0f}h after leak start')
print(f'   Projected crash time   : {(leak_start + timedelta(hours=hours_to_crash)).strftime("%Y-%m-%d %H:%M UTC")}')


## Step 6 - Visualise the Full 72-Hour Memory Profile

Before uploading, plot the complete time series.
This is what the ML model will analyse — and what the operations team
would have needed to spot manually without AIOps.

Pay attention to:
- How stable the baseline looks across 48 hours
- How gradual the leak is — hard to spot without comparison to the baseline
- The swap usage rising as memory pressure builds (secondary indicator)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

timestamps    = [datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00')) for e in events]
mem_values    = [e['system.memory.used.pct'] * 100 for e in events]
swap_values   = [e['system.memory.swap.used.pct'] * 100 for e in events]
cpu_values    = [e['system.cpu.total.pct'] * 100 for e in events]
phases        = [e['phase'] for e in events]

colors = ['#1B3A6B' if p == 'baseline' else '#C55A11' for p in phases]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Memory
ax1.scatter(timestamps, mem_values, c=colors, s=2, alpha=0.6, zorder=2)
ax1.axvline(leak_start, color='#C55A11', linestyle='--', linewidth=2,
            label='Leak begins (hour 48)')
ax1.axhline(CRASH_THRESHOLD * 100, color='red', linestyle=':', linewidth=1.5,
            label=f'Crash threshold ({CRASH_THRESHOLD*100:.0f}%)')
ax1.axhline(avg_baseline * 100, color='#0D6E6E', linestyle=':', linewidth=1.5,
            label=f'Baseline avg ({avg_baseline*100:.1f}%)')
ax1.set_ylabel('Memory Used %', fontsize=10)
ax1.set_title(
    f'Memory Utilisation - {SERVER_NAME}  |  72 hours: 48h stable baseline + 24h progressive leak',
    fontsize=11, fontweight='bold'
)
ax1.set_ylim(0, 105)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
baseline_patch = mpatches.Patch(color='#1B3A6B', label='Baseline (normal)')
leak_patch     = mpatches.Patch(color='#C55A11', label='Leak phase')
ax1.legend(handles=[baseline_patch, leak_patch], loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Panel 2: Swap
ax2.scatter(timestamps, swap_values, c=colors, s=2, alpha=0.5, zorder=2)
ax2.axvline(leak_start, color='#C55A11', linestyle='--', linewidth=2)
ax2.set_ylabel('Swap Used %', fontsize=10)
ax2.set_title('Swap Utilisation - Rises as RAM pressure builds (secondary anomaly signal)',
              fontsize=10)
ax2.set_ylim(0, 75)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax2.grid(True, alpha=0.3)

# Panel 3: CPU
ax3.scatter(timestamps, cpu_values, c=['#888888'] * len(cpu_values), s=2, alpha=0.4)
ax3.axvline(leak_start, color='#C55A11', linestyle='--', linewidth=2)
ax3.set_ylabel('CPU Total %', fontsize=10)
ax3.set_xlabel('Time (UTC)', fontsize=10)
ax3.set_title('CPU Utilisation - Slight upward trend under memory pressure (correlated signal)',
              fontsize=10)
ax3.set_ylim(0, 100)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax3.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print('Three-panel analysis:')
print('  Panel 1: Memory leak - gradual trend hard to spot without a baseline comparison')
print('  Panel 2: Swap rising - corroborating signal the ML can use alongside memory')
print('  Panel 3: CPU - slight increase under memory pressure, a third correlated signal')
print()
print('Key insight: a single-metric threshold alert on memory would not fire until ~90%.')
print('The ML detects the anomalous TREND hours earlier, while the value still looks acceptable.')


## Step 7 - Upload to Elastic Cloud

Uploading all data points in batches of 500.
The 72-hour dataset contains approximately 8,640 samples — expect this to take 30-60 seconds.


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(events):,} metric samples to index "{INDEX_NAME}" ...')

success, errors = bulk(
    es,
    generate_actions(events, INDEX_NAME),
    chunk_size=500,
    raise_on_error=False
)

print(f'Upload complete!')
print(f'   Documents indexed : {success:,}')
if errors:
    print(f'   Errors            : {len(errors)}')
    for e in errors[:3]: print(f'      {e}')
else:
    print(f'   Errors            : 0')

es.indices.refresh(index=INDEX_NAME)
print('Index refreshed - data is searchable in Kibana.')


## Step 8 - Verify the Upload


In [ ]:
total = es.count(index=INDEX_NAME)['count']

stats = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {
        'avg_mem':     {'avg': {'field': 'system.memory.used.pct'}},
        'max_mem':     {'max': {'field': 'system.memory.used.pct'}},
        'by_phase':    {'terms': {'field': 'phase', 'size': 5}},
        'avg_swap':    {'avg': {'field': 'system.memory.swap.used.pct'}},
    }
})

aggs = stats['aggregations']
print(f'Verification summary - index "{INDEX_NAME}":')
print(f'   Total samples   : {total:,}')
print(f'   Overall avg RAM : {aggs["avg_mem"]["value"]*100:.1f}%')
print(f'   Peak RAM        : {aggs["max_mem"]["value"]*100:.1f}%')
print(f'   Overall avg swap: {aggs["avg_swap"]["value"]*100:.1f}%')
print()
print('Breakdown by phase:')
for b in aggs['by_phase']['buckets']:
    print(f'   {b["key"]:10s}  {b["doc_count"]:>6,} samples')


## Step 9 - The Full AIOps Loop in Kibana

Follow these steps carefully. Each step represents one phase of the AIOps loop.

---

## OBSERVE: Create the Data View

1. Kibana > Discover > Create data view
   - Index pattern: `aiops-capstone`
   - Timestamp field: `@timestamp`
   - Save as: `AIOps Capstone`
2. In Discover, set the time range to **Last 3 days** (covers your full 72-hour window)
3. Add the column `system.memory.used.pct` to the Discover view
4. Observe: can you see the upward trend in the memory values over the last 24 hours?

---

## THINK: Create the ML Anomaly Detection Job

### 9a - Create the Job
1. Kibana > **Machine Learning > Anomaly Detection > Create Job**
2. Select data view: `aiops-capstone`
3. Job type: **Single Metric**
4. Configure:

| Setting | Value | Why |
|---------|-------|-----|
| Function | `high_mean` | We expect the anomaly to be an unusually HIGH value |
| Field | `system.memory.used.pct` | The metric we want to monitor |
| Bucket span | `1h` | Hourly buckets match the leak rate (+2%/hr) |
| Job ID | `capstone-memory-leak` | Descriptive name for the job |

5. Set time range to **Full** (all 72 hours)
6. Click **Create Job** and wait for it to complete (~30-60 seconds)

---

### 9b - Analyse Results in Anomaly Explorer
1. Machine Learning > **Anomaly Explorer**
2. Select job: `capstone-memory-leak`
3. Look at the swimlane heatmap:
   - The first 48 hours (baseline) should be blue/green (normal)
   - The last 24 hours (leak) should show amber/red blocks (anomalous)
4. Click on the first amber/red bucket and record:
   - **Anomaly score:** ___________
   - **Actual value:** ___________% RAM
   - **Typical value (what ML expected):** ___________% RAM
   - **Timestamp of first detection:** ___________

> **Key question:** At the time the ML first flagged an anomaly, what was the RAM value?
> Was it already near 90%? Or did the ML detect it much earlier?
> This is the difference between AIOps and a static threshold alert.

---

### 9c - Run a 24-Hour Forecast
1. Machine Learning > **Single Metric Viewer**
2. Select job: `capstone-memory-leak`
3. Click **Forecast** > Duration: **24 hours**
4. Record:
   - At what time does the forecast predict RAM will cross **90%**? ___________
   - How does this compare to the scenario's estimated crash time (see Step 2)? ___________
   - How wide is the confidence interval band? What does its width tell you? ___________

---

## ACT: Create the Automated Webhook Alert

### 9d - Set Up the Webhook Connector
1. Open **webhook.site** in a new browser tab
2. Copy your unique webhook URL (looks like: `https://webhook.site/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`)
3. Kibana > **Stack Management > Rules and Connectors > Connectors > Create connector**
4. Select: **Webhook**
5. Configure:
   - Name: `AIOps-Capstone-Webhook`
   - URL: paste your webhook.site URL
   - Method: `POST`
   - Header: `Content-Type` = `application/json`
6. Click **Save and test** - you should see a test POST appear at webhook.site

---

### 9e - Create the Alert Rule
1. **Rules and Connectors > Rules > Create Rule**
2. Rule type: **Anomaly detection alert**
3. Configure:
   - Job: `capstone-memory-leak`
   - Condition: anomaly score **> 75**
   - Check every: **15 minutes**
4. Action: select your `AIOps-Capstone-Webhook` connector
5. Request body - paste this JSON:
```json
{
  "server": "microland-app-03",
  "service": "java-middleware-svc",
  "issue": "Memory leak detected by ML anomaly detection",
  "anomaly_score": "{{context.anomalyScore}}",
  "metric": "system.memory.used.pct",
  "actual_value": "{{context.anomalyStartTimestamp}}",
  "severity": "High",
  "recommended_action": "Schedule controlled restart of java-middleware-svc during next maintenance window",
  "forecast": "Service will reach 90% RAM in approximately 18 hours at current leak rate"
}
```
6. Click **Save** to activate the rule

---

### 9f - Verify the Alert Fired
1. Go back to your **webhook.site** tab
2. You should see a POST request with the JSON payload above
3. If no request appears within 2 minutes:
   - Check that your ML job has completed and detected anomalies
   - Verify the rule is Active (not Disabled) in Rules and Connectors
   - Lower the threshold to 50 temporarily to trigger it manually

---

### Capstone Complete — Document Your Work

Write a **shift handover note** as if you are passing this incident to the next operations team.
Use the cell below as a template. Fill in the values from your Kibana results.


## Step 10 - Shift Handover Note (Fill In Your Results)

Run the cell below, then fill in the bracketed values based on what you observed in Kibana.
This mirrors a real shift handover document in an AIOps-enabled operations centre.


In [ ]:
from datetime import datetime, timezone

# Fill these in with your actual values from Kibana
FIRST_DETECTION_TIME   = 'FILL IN from Anomaly Explorer'   # e.g. '2025-05-20 14:00 UTC'
FIRST_ANOMALY_SCORE    = 'FILL IN'                          # e.g. '78'
FIRST_ANOMALY_RAM_PCT  = 'FILL IN'                          # e.g. '62.4%'
FORECAST_CRASH_TIME    = 'FILL IN from Forecast'            # e.g. '2025-05-21 08:00 UTC'
WEBHOOK_FIRED          = 'YES / NO'                         # Did the webhook fire?
RECOMMENDED_ACTION     = 'Schedule controlled restart of java-middleware-svc during next maintenance window'

handover = f"""
=======================================================================
SHIFT HANDOVER NOTE - AIOps Alert
Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}
=======================================================================

SERVER      : microland-app-03
SERVICE     : java-middleware-svc
ISSUE TYPE  : Progressive memory leak (ML anomaly detection)

DETECTION SUMMARY
  - First flagged by ML      : {FIRST_DETECTION_TIME}
  - Anomaly score at detection: {FIRST_ANOMALY_SCORE} / 100
  - RAM at first detection   : {FIRST_ANOMALY_RAM_PCT}
  - Leak rate                : +2% per hour (confirmed by trend analysis)
  - Forecast crash time      : {FORECAST_CRASH_TIME} (RAM crosses 90%)

AUTOMATED ACTIONS TAKEN
  - ML anomaly detection job : ACTIVE (capstone-memory-leak)
  - Webhook alert fired      : {WEBHOOK_FIRED}
  - Auto-ticket created      : NO (manual escalation required for this scenario)

RECOMMENDED ACTION FOR RECEIVING TEAM
  {RECOMMENDED_ACTION}

  Priority    : P3 (degrading, not yet impacting users)
  Urgency     : Action required within 12 hours
  Escalate to : Application Support Team

SUPPORTING EVIDENCE
  - Kibana ML job    : capstone-memory-leak (Anomaly Explorer)
  - Webhook payload  : webhook.site (see received POST)
  - Forecast chart   : Single Metric Viewer > Forecast (7-day view)

=======================================================================
"""

print(handover)


## Step 11 - Capstone Reflection Questions

Answer these questions in the text cell below before submitting your completed notebook.
These mirror the types of questions asked in the PeopleCert AIOps Foundation exam.


### Your Answers (double-click this cell to edit)

**Q1:** At what RAM percentage did the ML first detect the anomaly?
Was this before or after the value would have crossed a traditional static threshold of 85%?
What does this tell you about the advantage of probabilistic ML over deterministic thresholds?

> *Your answer here*

---

**Q2:** The full Observe-Think-Act loop you built today required:
Metricbeat (or simulated data) > Elasticsearch > ML job > Alert rule > Webhook.
Identify one point of failure in this chain and describe how you would make it resilient.

> *Your answer here*

---

**Q3:** In this scenario, the automated action was a webhook notification (inform only).
What would be required — technically and from a governance perspective — to upgrade this
to a fully automated 'self-healing' service restart instead of a notification?

> *Your answer here*

---

**Q4:** Which two of the 8 PeopleCert AIOps Foundation blueprint domains were most
directly demonstrated in this capstone lab? Explain your reasoning.

> *Your answer here*


## Optional Extension - Multi-Metric Anomaly Detection

The capstone used a single-metric job on memory. For a more realistic production scenario,
you would create a **multi-metric job** that analyses memory, CPU, and swap simultaneously.
When all three deviate together, the ML confidence is much higher.

### Create a Multi-Metric Job
1. Machine Learning > Anomaly Detection > Create Job > **Multi Metric**
2. Index: `aiops-capstone`
3. Add three detectors:
   - `high_mean(system.memory.used.pct)`
   - `high_mean(system.memory.swap.used.pct)`
   - `high_mean(system.cpu.total.pct)`
4. Split data by: `host.name` (so the model is per-server)
5. Job ID: `capstone-multi-metric`
6. Run the job and compare the anomaly scores to your single-metric job.

**Question:** Does the multi-metric job detect the leak earlier (lower anomaly score threshold
needed) or at the same time as the single-metric job? Why might this be?
